# 🚀 Fine-Tuning Nexus Universal AI Model (All Topics & Domains)
### Fast LoRA Training on Google Colab (Free T4 GPU)

This notebook fine-tunes **Llama-3.2-1B-Instruct** across multiple domains: Coding (Python, JS, C#, SQL), Science, Mathematics, Creative Problem Solving, and General Knowledge. Once trained, it pushes the weights to your **Hugging Face Hub**.

## 1. Install Unsloth & Required Libraries

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Load Base Model & Tokenizer (Llama 3.2 1B Instruct - 4bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

## 3. Attach LoRA Parameter-Efficient Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Load Dataset (`universal_ai_dataset.jsonl`)
Upload `universal_ai_dataset.jsonl` (from `E:\AI_Model_Project`) into the Colab Files explorer on the left, then run this cell.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Load universal dataset
dataset = load_dataset("json", data_files="universal_ai_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"Dataset loaded successfully with {len(dataset)} instruction pairs!")

## 5. Start Training (5 - 8 mins on Free T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("Training Complete!")

## 6. Test Universal Reasoning Live in Notebook

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are Nexus AI, a universal intelligence model. Answer with clarity and depth."},
    {"role": "user", "content": "Explain how neurons in the human brain communicate, and write a simple Python script to simulate a basic perceptron."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 350, use_cache = True)

## 7. Push Trained Weights to Hugging Face Hub

In [ ]:
from huggingface_hub import login, HfApi
login()

api = HfApi()
current_user = api.whoami()["name"]
REPO_NAME = f"{current_user}/nexus-ai-universal-lora"

print(f"Uploading model to: https://huggingface.co/{REPO_NAME} ...")
model.push_to_hub(REPO_NAME, token = True)
tokenizer.push_to_hub(REPO_NAME, token = True)

print(f"Model published live to: https://huggingface.co/{REPO_NAME}")
